In [1]:
import sys
sys.path.append('..')
from burauEnv import BurauEnv


In [2]:
import torch 
from torch import nn
from torch.functional import F
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class BurauSolver(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(BurauSolver, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, 4)
        self.input_dim = input_dim

    def forward(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(0)
        x = F.relu(self.fc1(x)) # x shape: (batch_size, hidden_dim)
        x = self.fc2(x) # logits shape: (batch_size, num_cities)
        return self.fc3(x)

    def sample_word(self,temp = 1):
        
        env = BurauEnv(2,self.input_dim)
        state, _ = env.reset()
        done = False
        total_reward = 0
        while not done:
            with torch.no_grad():
                probs = F.softmax(self.forward(torch.tensor(state, dtype=torch.float32).to(device)).squeeze(0)/temp, dim = 0)
                action = int(torch.multinomial(probs, num_samples=1).squeeze(-1))
                state, reward, term, trunc, _ = env.step(action)
                total_reward += reward
                done = term or trunc
        return env, total_reward

In [3]:
solver = BurauSolver(32,64)
sample = solver.sample_word()
print(sample[0].render(), sample[1])

[Turn 32] word = aBbabaAbBBbAAaabbaaababAbabaAbBa
None -84.4


In [10]:
import numpy as np
def cross_entropy_method(solver, optimizer,
                         num_iterations=10, population_size=100, elite_fraction=0.2):


    best_word = None
    best_reward = float('-inf')

    
    # Define loss function
    criterion = nn.CrossEntropyLoss()

    for iteration in range(num_iterations):
        # Generate a population of tours
        words = []
        word_rewards = []
        power_ranges = []
        temp = 1
        if(iteration > num_iterations//10):
            temp = 3
        for _ in range(population_size):
            word, word_reward = solver.sample_word(temp = temp) 
            
            words.append(word.word)
            power_ranges.append(word)
            word_rewards.append(word_reward)


        # Select elite tours
        num_elite = int(population_size * elite_fraction)
        elite_indices = np.argsort(word_rewards)[num_elite:]
        elite_words = [words[i] for i in elite_indices]

        worst_elite_tour=word_rewards[elite_indices[0]]

        # Update best solution
        current_best_reward = max(word_rewards)
        if current_best_reward > best_reward:
            best_reward = current_best_reward
            best_word = words[np.argmax(word_rewards)]
            best_power_range = power_ranges[np.argmax(word_rewards)]

        # Train the model on elite tours
        if elite_words:
            # Reset gradients
            solver.zero_grad()

            for word in elite_words:
                length = len(word)
                for j in range(length):
                    target_letter = torch.tensor(word[j]-1, dtype=torch.long).unsqueeze(0)
                    prefix_word = word.copy()
                    # Prepare input data
                    prefix_word[j:] = 0
                    
                    
                    output = solver(torch.tensor(prefix_word, dtype = torch.float32).to(device))



                    # Calculate loss
                    loss = criterion(output, target_letter)

                    # Backward pass and optimization
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

        if (iteration + 1) % 1 == 0:
            print(f"Iteration [{iteration+1}/{num_iterations}]")
            print(f"Worst Elite word: {worst_elite_tour}")
            print(f"Current Best word: {best_word}")
            best_power_range.render()
            print(f"Current word Power_Range: {best_power_range.power_range}")
            print(f"Current word Length: {best_reward}")

    return worst_elite_tour, best_word, best_reward

In [11]:
solver = BurauSolver(32,128)
optimizer = torch.optim.Adam(solver.parameters(),lr = 0.1)
cross_entropy_method(solver, optimizer,
                         num_iterations=200, population_size=50000, elite_fraction=0.01)

Iteration [1/200]
Worst Elite word: -150.70000000000002
Current Best word: [3 3 1 2 1 2 2 4 4 4 2 3 1 2 4 4 2 4 2 2 4 4 4 4 3 4 2 1 1 2 4 4]
[Turn 32] word = bbABABBaaaBbABaaBaBBaaaabaBAABaa
Current word Power_Range: 19
Current word Length: -27.99999999999999
Iteration [2/200]
Worst Elite word: -142.60000000000002
Current Best word: [1 2 2 4 2 2 2 4 2 2 2 2 2 2 4 3 1 1 2 2 2 4 4 2 2 2 2 2 2 2 2 2]
[Turn 32] word = ABBaBBBaBBBBBBabAABBBaaBBBBBBBBB
Current word Power_Range: 24
Current word Length: -23.200000000000003
Iteration [3/200]
Worst Elite word: -50
Current Best word: [2 2 2 2 2 4 2 2 2 2 2 2 2 2 2 2 4 2 2 2 2 2 2 2 2 2 2 4 2 2 2 2]
[Turn 32] word = BBBBBaBBBBBBBBBBaBBBBBBBBBBaBBBB
Current word Power_Range: 8
Current word Length: -7.200000000000003
Iteration [4/200]
Worst Elite word: -32
Current Best word: [2 2 2 2 2 4 2 2 2 2 2 2 2 2 2 2 4 2 2 2 2 2 2 2 2 2 2 4 2 2 2 2]
[Turn 32] word = BBBBBaBBBBBBBBBBaBBBBBBBBBBaBBBB
Current word Power_Range: 8
Current word Length: -7.200000000

KeyboardInterrupt: 

In [ ]:
cross_entropy_method(solver, optimizer,
                         num_iterations=200, population_size=100000, elite_fraction=0.01)

Iteration [1/200]
Worst Elite word: -28.699999999999996
Current Best word: [2 2 4 2 2 2 2 4 2 2 2 2 2 4 2 2 2 2 4 2 2 2 2 4 2 2 2 2 2 4 2 2]
[Turn 32] word = BBaBBBBaBBBBBaBBBBaBBBBaBBBBBaBB
Current word Power_Range: 6
Current word Length: -5.000000000000002
Iteration [2/200]
Worst Elite word: -29.799999999999997
Current Best word: [2 2 4 2 2 2 4 2 2 2 2 2 4 2 2 2 2 4 2 2 2 2 4 2 2 2 2 2 4 2 2 2]
[Turn 32] word = BBaBBBaBBBBBaBBBBaBBBBaBBBBBaBBB
Current word Power_Range: 6
Current word Length: -4.200000000000005
Iteration [3/200]
Worst Elite word: -29.799999999999997
Current Best word: [2 2 2 4 2 2 2 2 4 2 2 2 4 2 2 2 2 4 2 2 2 4 2 2 2 4 2 2 2 2 4 2]
[Turn 32] word = BBBaBBBBaBBBaBBBBaBBBaBBBaBBBBaB
Current word Power_Range: 6
Current word Length: -3.800000000000006
Iteration [4/200]
Worst Elite word: -28.7
Current Best word: [2 2 2 4 2 2 2 2 4 2 2 2 4 2 2 2 2 4 2 2 2 4 2 2 2 4 2 2 2 2 4 2]
[Turn 32] word = BBBaBBBBaBBBaBBBBaBBBaBBBaBBBBaB
Current word Power_Range: 6
Current word Lengt